<a href="https://colab.research.google.com/github/prabhatdawn7-ship-it/NDVI_animation-google-earth-engine/blob/main/Ex_06_pg_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project = 'shruti-ee')

In [40]:
Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [42]:
India = ee.FeatureCollection("projects/shruti-ee/assets/National_Boundary")

In [43]:
ndvi = ee.ImageCollection('MODIS/006/MOD13A2').select('NDVI');
Map

Map(bottom=812.0, center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [44]:
region = ee.Geometry.Polygon(
    [[[67.060547, 4.740675],
     [67.060547, 38.065392],
     [98.964844, 38.065392],
     [98.964844, 4.740675],
     [67.060547, 4.740675]]],
    None,
    False)
#print(region.getInfo())

In [45]:
ndvi_india = ndvi.map(lambda img: img.set('doy',
ee.Date(img.get('system:time_start')).getRelative('day', 'year')))

In [46]:
distinctDOY = ndvi_india.filterDate('2000-02-18', '2002-02-18');

In [47]:
filter = ee.Filter.equals(leftField='doy', rightField='doy');
join = ee.Join.saveAll('doy_matches');
joinCol = ee.ImageCollection(join.apply(distinctDOY, ndvi_india, filter));

In [48]:
comp = joinCol.map(lambda img: ee.ImageCollection.fromImages(img.get('doy_matches')).reduce(ee.Reducer.mean()).set('doy', ee.Number(img.get('doy'))))

In [49]:
visParams = {
  'min': -2000.0,
  'max': 10000.0,
  'palette': [
    'FFFFFF', 'CE7E45', 'DF923D', 'F1B555', 'FCD163', '99B718', '74A901',
    '66A000', '529400', '3E8601', '207401', '056201', '004C00', '023B01',
    '012E01', '011D01', '011301'
  ],
};

In [50]:
rgbVis = comp.map(lambda img: img.visualize(bands=['NDVI_mean'], **visParams).clip(India))

In [56]:
gifParams = {
  'region': region,
  'dimensions': 600,
  'projection': 'EPSG:4326',
  'framesPerSecond': 10,
  'format': 'gif'
};

In [52]:
print(rgbVis.getVideoThumbURL(gifParams));


https://earthengine.googleapis.com/v1/projects/shruti-ee/videoThumbnails/170f40fb6e7c4ba123ef18376cccc7e8-5f798f31c96448b19d67e8bf42067fd0:getPixels


In [53]:
import os

In [57]:
out_dir = '/content/drive/MyDrive/GEE_Exports/EX6-NDVI_TimeSeries'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)
out_gif = os.path.join(out_dir, 'india_ndvi_animation.gif')
geemap.download_ee_video(rgbVis, gifParams, out_gif)

Generating URL...
Please wait ...
The GIF image has been saved to: /content/drive/MyDrive/GEE_Exports/EX6-NDVI_TimeSeries/india_ndvi_animation.gif
